# Revue du dataset

Description des fichiers presents dans `data/` : apercu (head) et statistiques de base pour chaque fichier parquet.

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

DATA_DIR = Path("data")

df_patient = pd.read_parquet(DATA_DIR / "patient.parquet")
df_sejour = pd.read_parquet(DATA_DIR / "sejour.parquet")
df_mouvement = pd.read_parquet(DATA_DIR / "mouvement.parquet")
df_document = pd.read_parquet(DATA_DIR / "document.parquet")
df_biologie = pd.read_parquet(DATA_DIR / "biologie.parquet")
df_medicament = pd.read_parquet(DATA_DIR / "medicament.parquet")

fichiers = {
    "patient": df_patient,
    "sejour": df_sejour,
    "mouvement": df_mouvement,
    "document": df_document,
    "biologie": df_biologie,
    "medicament": df_medicament,
}

## Vue d'ensemble

Nombre de lignes / colonnes par fichier.

In [2]:
resume = pd.DataFrame({
    "fichier": list(fichiers.keys()),
    "n_lignes": [df.shape[0] for df in fichiers.values()],
    "n_colonnes": [df.shape[1] for df in fichiers.values()],
    "colonnes": [list(df.columns) for df in fichiers.values()],
})
resume

,fichier,n_lignes,n_colonnes,colonnes
0,patient,100,4,"[id_patient, age, sexe, date_deces]"
1,sejour,100,7,"[id_sejour, id_patient, date_entree, date_sort..."
2,mouvement,100,5,"[id_mouvement, id_sejour, uf, date_entree, dat..."
3,document,158,6,"[id_entrepot, id_patient, id_sejour, titre, ty..."
4,biologie,487,7,"[id_biologie, id_patient, id_sejour, uf, date_..."
5,medicament,272,8,"[id_medicament, id_patient, id_sejour, date_ad..."


## 1. `patient.parquet`

Une ligne par patient : age, sexe, date de deces eventuelle.

In [3]:
df_patient.head()

,id_patient,age,sexe,date_deces
0,PAT000001,62,M,NaT
1,PAT000002,57,F,NaT
2,PAT000003,37,F,NaT
3,PAT000004,66,F,NaT
4,PAT000005,16,M,NaT


In [4]:
df_patient.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   id_patient  100 non-null    object        
 1   age         100 non-null    int64         
 2   sexe        100 non-null    object        
 3   date_deces  1 non-null      datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(2)
memory usage: 3.2+ KB


In [5]:
df_patient.describe(include="all")

,id_patient,age,sexe,date_deces
count,100,100.000000,100,1
unique,100,NaN,2,NaN
top,PAT000001,NaN,M,NaN
freq,1,NaN,53,NaN
mean,NaN,54.550000,NaN,2024-04-07 08:08:26
min,NaN,5.000000,NaN,2024-04-07 08:08:26
25%,NaN,39.000000,NaN,2024-04-07 08:08:26
50%,NaN,56.000000,NaN,2024-04-07 08:08:26
75%,NaN,71.500000,NaN,2024-04-07 08:08:26
max,NaN,85.000000,NaN,2024-04-07 08:08:26


In [6]:
print("Valeurs manquantes par colonne :")
df_patient.isna().sum()

Valeurs manquantes par colonne :


id_patient     0
age            0
sexe           0
date_deces    99
dtype: int64

In [7]:
print("Repartition du sexe :")
print(df_patient["sexe"].value_counts())
print()
print(f"Part de patients decedes : {df_patient['date_deces'].notna().mean():.1%}")

Repartition du sexe :
sexe
M    53
F    47
Name: count, dtype: int64

Part de patients decedes : 1.0%


## 2. `sejour.parquet`

Une ligne par sejour hospitalier : dates d'entree/sortie, UF d'entree/sortie.

In [8]:
df_sejour.head()

,id_sejour,id_patient,date_entree,date_sortie,uf_entree,uf_sortie,specialite
0,SEJ000001,PAT000001,2023-12-25 14:40:46,2023-12-31 14:40:46,1710,1710,MEDECINE INTERNE
1,SEJ000002,PAT000002,2023-08-13 12:25:29,2023-08-15 12:25:29,1130,1130,CANCERO ADULTE
2,SEJ000003,PAT000003,2024-09-04 21:47:33,2024-09-11 21:47:33,1850,1850,OBSTETRIQUE
3,SEJ000004,PAT000004,2024-07-20 02:11:11,2024-07-29 02:11:11,1570,1570,HEPATO-GASTRO-ENTERO
4,SEJ000005,PAT000005,2024-06-16 09:51:21,2024-06-19 09:51:21,1940,1940,PNEUMOLOGIE


In [9]:
df_sejour.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   id_sejour    100 non-null    object        
 1   id_patient   100 non-null    object        
 2   date_entree  100 non-null    datetime64[ns]
 3   date_sortie  100 non-null    datetime64[ns]
 4   uf_entree    100 non-null    object        
 5   uf_sortie    100 non-null    object        
 6   specialite   100 non-null    object        
dtypes: datetime64[ns](2), object(5)
memory usage: 5.6+ KB


In [10]:
duree_sejour = (df_sejour["date_sortie"] - df_sejour["date_entree"]).dt.days
print("Duree de sejour (jours) :")
duree_sejour.describe()

Duree de sejour (jours) :


count    100.000000
mean       8.700000
std        8.703303
min        1.000000
25%        2.000000
50%        5.000000
75%       12.000000
max       36.000000
dtype: float64

In [11]:
print(f"Nombre de sejours par patient (moyenne) : {df_sejour.groupby('id_patient').size().mean():.2f}")
print()
print("Sejours par UF d'entree :")
df_sejour["uf_entree"].value_counts()

Nombre de sejours par patient (moyenne) : 1.00

Sejours par UF d'entree :


uf_entree
1130    14
1350    10
1940     9
1440     9
1240     9
1520     8
1710     6
1850     5
1570     5
1800     4
1610     4
1640     3
1990     3
1760     3
1960     2
1730     2
1150     2
1650     1
1250     1
Name: count, dtype: int64

## 3. `mouvement.parquet`

Une ligne par mouvement intra-sejour (changement d'UF).

In [12]:
df_mouvement.head()

,id_mouvement,id_sejour,uf,date_entree,date_sortie
0,MVT000001,SEJ000001,1710,2023-12-25 14:40:46,2023-12-31 14:40:46
1,MVT000002,SEJ000002,1130,2023-08-13 12:25:29,2023-08-15 12:25:29
2,MVT000003,SEJ000003,1850,2024-09-04 21:47:33,2024-09-11 21:47:33
3,MVT000004,SEJ000004,1570,2024-07-20 02:11:11,2024-07-29 02:11:11
4,MVT000005,SEJ000005,1940,2024-06-16 09:51:21,2024-06-19 09:51:21


In [13]:
df_mouvement.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   id_mouvement  100 non-null    object        
 1   id_sejour     100 non-null    object        
 2   uf            100 non-null    object        
 3   date_entree   100 non-null    datetime64[ns]
 4   date_sortie   100 non-null    datetime64[ns]
dtypes: datetime64[ns](2), object(3)
memory usage: 4.0+ KB


In [14]:
print(f"Nombre de mouvements par sejour (moyenne) : {df_mouvement.groupby('id_sejour').size().mean():.2f}")
print()
print("Mouvements par UF :")
df_mouvement["uf"].value_counts()

Nombre de mouvements par sejour (moyenne) : 1.00

Mouvements par UF :


uf
1130    14
1350    10
1940     9
1440     9
1240     9
1520     8
1710     6
1850     5
1570     5
1800     4
1610     4
1640     3
1990     3
1760     3
1960     2
1730     2
1150     2
1650     1
1250     1
Name: count, dtype: int64

## 4. `document.parquet`

Une ligne par document (compte-rendu, courrier, etc.) rattache a un sejour.

In [15]:
df_document.drop(columns=["texte_affichage"]).head()

,id_entrepot,id_patient,id_sejour,titre,type_document
0,DOC000001,PAT000001,SEJ000001,Compte-rendu d'hospitalisation - 2023-12-31,Compte-rendu d'hospitalisation
1,DOC000002,PAT000002,SEJ000002,Compte-rendu d'hospitalisation - 2023-08-15,Compte-rendu d'hospitalisation
2,DOC000003,PAT000003,SEJ000003,Compte-rendu d'hospitalisation - 2024-09-11,Compte-rendu d'hospitalisation
3,DOC000004,PAT000004,SEJ000004,Compte-rendu d'hospitalisation - 2024-07-29,Compte-rendu d'hospitalisation
4,DOC000005,PAT000005,SEJ000005,Compte-rendu d'hospitalisation - 2024-06-19,Compte-rendu d'hospitalisation


In [16]:
df_document.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 158 entries, 0 to 157
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id_entrepot      158 non-null    object
 1   id_patient       158 non-null    object
 2   id_sejour        158 non-null    object
 3   titre            158 non-null    object
 4   type_document    158 non-null    object
 5   texte_affichage  158 non-null    object
dtypes: object(6)
memory usage: 7.5+ KB


In [17]:
print("Repartition par type de document :")
print(df_document["type_document"].value_counts())
print()
longueur_texte = df_document["texte_affichage"].str.len()
print("Longueur du texte HTML (caracteres) :")
longueur_texte.describe()

Repartition par type de document :
type_document
Compte-rendu d'hospitalisation    100
Compte-rendu de consultation       29
Compte-rendu operatoire            29
Name: count, dtype: int64

Longueur du texte HTML (caracteres) :


count      158.000000
mean      2903.797468
std       2163.151693
min        368.000000
25%       1316.000000
50%       2466.500000
75%       3796.250000
max      13108.000000
Name: texte_affichage, dtype: float64

## Exemple de contenu d'un document

In [18]:
from IPython.display import HTML
HTML(df_document.iloc[0]["texte_affichage"])

## 5. `biologie.parquet`

Une ligne par resultat de biologie rattache a un sejour : valeur numerique ou textuelle selon le type de resultat.

In [19]:
df_biologie.head()

,id_biologie,id_patient,id_sejour,uf,date_prelevement,valeur_numerique,valeur_texte
0,BIO000001,PAT000001,SEJ000001,1710,2023-12-30 02:40:32,55.08,None
1,BIO000002,PAT000001,SEJ000001,1710,2023-12-30 01:56:09,20.59,None
2,BIO000003,PAT000001,SEJ000001,1710,2023-12-28 22:06:15,17.48,None
3,BIO000004,PAT000001,SEJ000001,1710,2023-12-25 19:01:08,18.83,None
4,BIO000005,PAT000001,SEJ000001,1710,2023-12-29 06:21:17,5.40,None


In [20]:
df_biologie.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 487 entries, 0 to 486
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   id_biologie       487 non-null    object        
 1   id_patient        487 non-null    object        
 2   id_sejour         487 non-null    object        
 3   uf                487 non-null    object        
 4   date_prelevement  487 non-null    datetime64[ns]
 5   valeur_numerique  364 non-null    float64       
 6   valeur_texte      123 non-null    object        
dtypes: datetime64[ns](1), float64(1), object(5)
memory usage: 26.8+ KB


In [21]:
print(f"Nombre de resultats de biologie par sejour (moyenne) : {df_biologie.groupby('id_sejour').size().mean():.2f}")
print()
n_numerique = df_biologie["valeur_numerique"].notna().sum()
n_texte = df_biologie["valeur_texte"].notna().sum()
print(f"Resultats numeriques : {n_numerique} ({n_numerique / len(df_biologie):.1%})")
print(f"Resultats textuels   : {n_texte} ({n_texte / len(df_biologie):.1%})")
print()
print("Repartition des valeurs textuelles :")
print(df_biologie["valeur_texte"].value_counts())

Nombre de resultats de biologie par sejour (moyenne) : 4.87

Resultats numeriques : 364 (74.7%)
Resultats textuels   : 123 (25.3%)

Repartition des valeurs textuelles :
valeur_texte
non contributif    34
anormal            27
normal             26
positif            19
negatif            17
Name: count, dtype: int64


## 6. `medicament.parquet`

Une ligne par administration medicamenteuse rattachee a un sejour (code UCD/ATC, quantite, unite).

In [22]:
df_medicament.head()

,id_medicament,id_patient,id_sejour,date_administration,quantite_administree,unite,ucd,atc
0,MED000001,PAT000001,SEJ000001,2023-12-29 20:21:12,716,mg,3400892014319,J01CA04
1,MED000002,PAT000002,SEJ000002,2023-08-13 18:09:45,782,mg,3400935928562,A10BA02
2,MED000003,PAT000003,SEJ000003,2024-09-09 08:03:27,4,mg,3400892449517,N05BA01
3,MED000004,PAT000003,SEJ000003,2024-09-09 05:16:21,11,mg,3400937246430,C09AA02
4,MED000005,PAT000003,SEJ000003,2024-09-05 18:35:23,230,mg,3400921959219,B01AC06


In [23]:
df_medicament.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 272 entries, 0 to 271
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   id_medicament         272 non-null    object        
 1   id_patient            272 non-null    object        
 2   id_sejour             272 non-null    object        
 3   date_administration   272 non-null    datetime64[ns]
 4   quantite_administree  272 non-null    int64         
 5   unite                 272 non-null    object        
 6   ucd                   272 non-null    object        
 7   atc                   272 non-null    object        
dtypes: datetime64[ns](1), int64(1), object(6)
memory usage: 17.1+ KB


In [24]:
print(f"Nombre d'administrations par sejour (moyenne) : {df_medicament.groupby('id_sejour').size().mean():.2f}")
print()
print("Repartition par code ATC :")
print(df_medicament["atc"].value_counts())

Nombre d'administrations par sejour (moyenne) : 2.72

Repartition par code ATC :
atc
N05BA01    39
C10AA05    29
N02AA01    28
J01CA04    28
M01AE01    27
B01AC06    26
A02BC01    25
A10BA02    24
N02BE01    24
C09AA02    22
Name: count, dtype: int64
